[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/03_ONNX_IR_Specification/ONNX_IR_Specification_Apply.ipynb)

# 3.3 ONNX IR Specification — Hands-On Practice

## Objective

Walk the **Protobuf hierarchy** of ONNX's Intermediate Representation (IR):
`ModelProto → GraphProto → NodeProto → TensorProto → AttributeProto`.
Decode attributes, test opset compatibility, and understand IR versions.

---

| # | Section | Focus |
|---|---------|-------|
| 1 | Setup | Dependencies |
| 2 | Exercise 1: Proto Hierarchy Walk | Traverse ModelProto tree |
| 3 | Exercise 2: Attribute Decoding | Read all attribute types |
| 4 | Exercise 3: IR Version Analysis | How IR versions changed |
| 5 | Exercise 4: Opset Compatibility | Test operators across opsets |
| 6 | Exercise 5: TensorProto Formats | Raw data vs float_data |
| 7 | Exercise 6: Graph vs Subgraph | Compare main graph and subgraphs |
| 8 | Challenge: Proto-to-Dict Serializer | Convert any model to a dict |

In [ ]:
# !pip install onnx onnxruntime numpy

import numpy as np
import onnx
from onnx import TensorProto, AttributeProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid,
    make_tensor, printable_graph)
from onnx.checker import check_model
from onnx.numpy_helper import from_array, to_array
import onnxruntime as ort

print(f'ONNX: {onnx.__version__}  IR: {onnx.IR_VERSION}  ORT: {ort.__version__}')

## Exercise 1: Proto Hierarchy Walk

ONNX's IR is a tree of Protobuf messages:

```
ModelProto
├── ir_version, opset_import[], producer_name, ...
├── GraphProto
│   ├── name, input[], output[], initializer[]
│   ├── NodeProto[]
│   │   ├── op_type, input[], output[]
│   │   └── AttributeProto[]
│   │       ├── name, type
│   │       └── value (f, i, s, t, g, ...)
│   └── value_info[]
│       └── TypeProto (elem_type, shape)
└── functions[]
```

In [ ]:
# Build a model with various features
np.random.seed(42)
W = from_array(np.random.randn(4, 3).astype(np.float32), 'W')
b = from_array(np.zeros(3, dtype=np.float32), 'b')

X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 3])

nodes = [
    make_node('MatMul', ['X', 'W'], ['XW']),
    make_node('Add', ['XW', 'b'], ['Z']),
    make_node('Relu', ['Z'], ['Y']),
]
g = make_graph(nodes, 'demo', [X], [Y], [W, b])
model = make_model(g, opset_imports=[make_opsetid('', 17)])
model.producer_name = 'ONNX Tutorial'
model.producer_version = '1.0'
model.doc_string = 'Demo model for IR exploration'
model.model_version = 1
check_model(model)

def walk_proto(model):
    """Walk the full proto hierarchy and print structure."""
    print('ModelProto:')
    print(f'  ir_version:       {model.ir_version}')
    print(f'  producer_name:    {model.producer_name}')
    print(f'  producer_version: {model.producer_version}')
    print(f'  model_version:    {model.model_version}')
    print(f'  doc_string:       {model.doc_string[:50]}...' if len(model.doc_string) > 50 else f'  doc_string:       {model.doc_string}')

    print(f'  opset_import ({len(model.opset_import)}):')
    for oi in model.opset_import:
        d = oi.domain or 'ai.onnx'
        print(f'    domain="{d}"  version={oi.version}')

    g = model.graph
    print(f'\n  GraphProto: "{g.name}"')
    print(f'    inputs ({len(g.input)}):')
    for inp in g.input:
        t = inp.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'      {inp.name}: {dtype} {dims}')

    print(f'    outputs ({len(g.output)}):')
    for out in g.output:
        t = out.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'      {out.name}: {dtype} {dims}')

    print(f'    initializers ({len(g.initializer)}):')
    for init in g.initializer:
        arr = to_array(init)
        print(f'      {init.name}: shape={list(arr.shape)} dtype={arr.dtype}')

    print(f'    nodes ({len(g.node)}):')
    for i, node in enumerate(g.node):
        print(f'      [{i}] {node.op_type}: '
              f'{list(node.input)} → {list(node.output)}')
        for attr in node.attribute:
            print(f'          attr "{attr.name}" type={attr.type}')

    if model.functions:
        print(f'  functions ({len(model.functions)}):')
        for f in model.functions:
            print(f'    {f.domain}::{f.name}')

walk_proto(model)

## Exercise 2: Attribute Decoding

ONNX `AttributeProto` has these types:

| Type | Enum | Python Field |
|------|------|--------------|
| FLOAT | 1 | `attr.f` |
| INT | 2 | `attr.i` |
| STRING | 3 | `attr.s` |
| TENSOR | 4 | `attr.t` |
| GRAPH | 5 | `attr.g` |
| FLOATS | 6 | `attr.floats` |
| INTS | 7 | `attr.ints` |
| STRINGS | 8 | `attr.strings` |

In [ ]:
# Build a model with many attribute types
W_c = from_array(np.random.randn(8, 3, 3, 3).astype(np.float32) * 0.1, 'W')
bn_s = from_array(np.ones(8, dtype=np.float32), 's')
bn_b = from_array(np.zeros(8, dtype=np.float32), 'b')
bn_m = from_array(np.zeros(8, dtype=np.float32), 'm')
bn_v = from_array(np.ones(8, dtype=np.float32), 'v')

X = make_tensor_value_info('X', TensorProto.FLOAT, [1, 3, 16, 16])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, None)

attr_model = make_model(
    make_graph([
        make_node('Conv', ['X', 'W'], ['c'],
                 kernel_shape=[3, 3], strides=[1, 1],
                 pads=[1, 1, 1, 1], group=1),
        make_node('BatchNormalization',
                 ['c', 's', 'b', 'm', 'v'], ['bn'], epsilon=1e-5),
        make_node('Relu', ['bn'], ['Y']),
    ], 'attr_demo', [X], [Y], [W_c, bn_s, bn_b, bn_m, bn_v]),
    opset_imports=[make_opsetid('', 17)])

def decode_attribute(attr):
    """Decode any ONNX attribute to Python."""
    type_names = {
        1: 'FLOAT', 2: 'INT', 3: 'STRING', 4: 'TENSOR',
        5: 'GRAPH', 6: 'SPARSE_TENSOR',
        7: 'FLOATS', 8: 'INTS', 9: 'STRINGS',
        10: 'TENSORS', 11: 'GRAPHS',
    }
    decoders = {
        1: lambda a: a.f,
        2: lambda a: a.i,
        3: lambda a: a.s.decode('utf-8') if a.s else '',
        4: lambda a: f'Tensor{list(to_array(a.t).shape)}',
        5: lambda a: f'Graph("{a.g.name}", {len(a.g.node)} nodes)',
        7: lambda a: list(a.floats),
        8: lambda a: list(a.ints),
        9: lambda a: [s.decode('utf-8') for s in a.strings],
    }
    type_name = type_names.get(attr.type, f'UNKNOWN({attr.type})')
    decoder = decoders.get(attr.type, lambda a: '<?>')
    value = decoder(attr)
    return type_name, value

print('All attributes in the model:')
print(f'{"Node":10s} {"Attr Name":15s} {"Type":10s} {"Value"}')
print('-' * 60)
for node in attr_model.graph.node:
    for attr in node.attribute:
        tname, val = decode_attribute(attr)
        print(f'  {node.op_type:10s} {attr.name:15s} {tname:10s} {val}')

## Exercise 3: IR Version Analysis

The ONNX IR version tracks changes to the protobuf schema itself.

In [ ]:
ir_history = {
    3: 'Initial stable IR',
    4: 'Added type_constraint validation',
    5: 'Subgraph support (If, Loop)',
    6: 'Training-related fields',
    7: 'Functions, Sequence/Map/Optional types',
    8: 'SparseTensor, Float8 types',
    9: 'Extended type system',
    10: 'Latest features',
}

current_ir = onnx.IR_VERSION
print(f'Current IR version: {current_ir}')
print(f'\nIR Version History:')
print('=' * 55)
for v, desc in ir_history.items():
    marker = ' ◄── current' if v == current_ir else ''
    print(f'  IR {v}: {desc}{marker}')

# Build models and check what IR version they get
for opset in [9, 11, 13, 15, 17, 19]:
    try:
        X = make_tensor_value_info('X', TensorProto.FLOAT, [1])
        Y = make_tensor_value_info('Y', TensorProto.FLOAT, [1])
        g = make_graph([make_node('Relu', ['X'], ['Y'])], 'ir', [X], [Y])
        m = make_model(g, opset_imports=[make_opsetid('', opset)])
        print(f'  opset {opset:2d} → IR version {m.ir_version}')
    except Exception as e:
        print(f'  opset {opset:2d} → error: {str(e)[:40]}')

## Exercise 4: Opset Compatibility Testing

Test if a specific operator combination works at different opset levels.

In [ ]:
from onnx.defs import get_schema, onnx_opset_version

def test_opset_compatibility(ops, opset_range=None):
    """Test operator combinations across opset versions."""
    if opset_range is None:
        opset_range = range(7, min(onnx_opset_version() + 1, 22))

    results = {}
    for v in opset_range:
        all_ok = True
        for op in ops:
            try:
                get_schema(op, v, '')
            except Exception:
                all_ok = False
                break
        results[v] = all_ok

    op_str = ' + '.join(ops)
    print(f'Compatibility: [{op_str}]')
    ok_versions = [v for v, ok in results.items() if ok]
    if ok_versions:
        print(f'  Supported: opset {min(ok_versions)} - {max(ok_versions)}')
    else:
        print(f'  Not supported in tested range')

    return results

# Common operator combos
combos = [
    ['MatMul', 'Add', 'Relu'],
    ['Conv', 'BatchNormalization', 'Relu'],
    ['LayerNormalization', 'MatMul'],
    ['Softmax', 'ArgMax'],
]

for combo in combos:
    test_opset_compatibility(combo)
    print()

## Exercise 5: TensorProto Data Formats

ONNX stores tensor data in two formats: typed arrays (`float_data`, `int32_data`, etc.)
or raw bytes (`raw_data`). Let's compare both.

In [ ]:
# Create tensors in both formats
data = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float32)

# Format 1: from_array uses raw_data
t1 = from_array(data, name='raw_format')

# Format 2: make_tensor uses float_data
t2 = make_tensor('typed_format', TensorProto.FLOAT, [4], data.tolist())

print('TensorProto Data Formats:')
print('=' * 50)

for label, t in [('from_array (raw)', t1), ('make_tensor (typed)', t2)]:
    has_raw = len(t.raw_data) > 0
    has_typed = len(t.float_data) > 0
    arr = to_array(t)
    print(f'\n  {label}:')
    print(f'    name:       {t.name}')
    print(f'    dims:       {list(t.dims)}')
    print(f'    data_type:  {TensorProto.DataType.Name(t.data_type)}')
    print(f'    raw_data:   {len(t.raw_data)} bytes  (present={has_raw})')
    print(f'    float_data: {len(t.float_data)} items (present={has_typed})')
    print(f'    decoded:    {arr}')
    assert np.allclose(arr, data)

# Size comparison
print(f'\nSerialized sizes:')
print(f'  raw_data format:   {len(t1.SerializeToString())} bytes')
print(f'  float_data format: {len(t2.SerializeToString())} bytes')

## Exercise 6: Main Graph vs Subgraph

Build a model with an `If` node and compare the main graph structure
with its embedded subgraphs.

In [ ]:
# Then branch: X + 1
t_out = make_tensor_value_info('then_out', TensorProto.FLOAT, [None])
one = from_array(np.array([1.0], dtype=np.float32), name='one')
then_g = make_graph(
    [make_node('Add', ['X', 'one'], ['then_out'])],
    'then_branch', [], [t_out], [one])

# Else branch: X * 2
e_out = make_tensor_value_info('else_out', TensorProto.FLOAT, [None])
two = from_array(np.array([2.0], dtype=np.float32), name='two')
else_g = make_graph(
    [make_node('Mul', ['X', 'two'], ['else_out'])],
    'else_branch', [], [e_out], [two])

X_main = make_tensor_value_info('X', TensorProto.FLOAT, [None])
cond = make_tensor_value_info('cond', TensorProto.BOOL, [])
Y_main = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

main = make_graph(
    [make_node('If', ['cond'], ['Y'],
              then_branch=then_g, else_branch=else_g)],
    'if_model', [X_main, cond], [Y_main])
model_if = make_model(main, opset_imports=[make_opsetid('', 17)])
check_model(model_if)

def compare_graph_levels(model):
    """Compare main graph and subgraphs."""
    print(f'{"Property":20s} {"Main":>10s}', end='')
    subgraphs = []
    for node in model.graph.node:
        for attr in node.attribute:
            if attr.type == AttributeProto.GRAPH:
                subgraphs.append((attr.name, attr.g))
                print(f' {attr.name:>10s}', end='')
    print()
    print('-' * (22 + 12 * (1 + len(subgraphs))))

    metrics = [
        ('name', lambda g: g.name),
        ('nodes', lambda g: len(g.node)),
        ('inputs', lambda g: len(g.input)),
        ('outputs', lambda g: len(g.output)),
        ('initializers', lambda g: len(g.initializer)),
    ]

    for label, fn in metrics:
        print(f'  {label:18s} {str(fn(model.graph)):>10s}', end='')
        for _, sg in subgraphs:
            print(f' {str(fn(sg)):>10s}', end='')
        print()

compare_graph_levels(model_if)

# Verify it runs
sess_if = ort.InferenceSession(
    model_if.SerializeToString(), providers=['CPUExecutionProvider'])
x = np.array([5.0, 10.0], dtype=np.float32)
r_true = sess_if.run(None, {'X': x, 'cond': np.array(True)})[0]
r_false = sess_if.run(None, {'X': x, 'cond': np.array(False)})[0]
print(f'\ncond=True:  {x} + 1 = {r_true}')
print(f'cond=False: {x} * 2 = {r_false}')

## Challenge: Proto-to-Dict Serializer

Build a function that converts any ONNX ModelProto into a plain Python
dictionary — useful for JSON export, logging, or model comparison.

In [ ]:
def model_to_dict(model):
    """Convert ModelProto to a nested dictionary."""
    def graph_to_dict(g):
        return {
            'name': g.name,
            'inputs': [
                {'name': i.name,
                 'type': TensorProto.DataType.Name(i.type.tensor_type.elem_type),
                 'shape': [d.dim_param or d.dim_value
                           for d in i.type.tensor_type.shape.dim]
                 if i.type.tensor_type.HasField('shape') else None}
                for i in g.input
            ],
            'outputs': [
                {'name': o.name}
                for o in g.output
            ],
            'initializers': [
                {'name': i.name,
                 'shape': list(i.dims),
                 'dtype': TensorProto.DataType.Name(i.data_type)}
                for i in g.initializer
            ],
            'nodes': [
                {
                    'op_type': n.op_type,
                    'inputs': list(n.input),
                    'outputs': list(n.output),
                    'attributes': {
                        a.name: decode_attribute(a)[1]
                        for a in n.attribute
                        if a.type not in (5, 11)  # skip graphs
                    }
                }
                for n in g.node
            ],
        }

    return {
        'ir_version': model.ir_version,
        'opset': [
            {'domain': oi.domain or 'ai.onnx', 'version': oi.version}
            for oi in model.opset_import
        ],
        'producer_name': model.producer_name,
        'producer_version': model.producer_version,
        'model_version': model.model_version,
        'doc_string': model.doc_string,
        'graph': graph_to_dict(model.graph),
        'metadata': {
            p.key: p.value for p in model.metadata_props
        },
    }

d = model_to_dict(model)

import json
print(json.dumps(d, indent=2, default=str)[:2000])
print('...')
print(f'\nDict keys: {list(d.keys())}')
print(f'Graph keys: {list(d["graph"].keys())}')
print(f'Nodes: {len(d["graph"]["nodes"])}')

---

## Summary

| Exercise | Topic | Key Takeaway |
|----------|-------|-------------|
| 1 | Proto hierarchy | ModelProto → GraphProto → NodeProto |
| 2 | Attribute decoding | FLOAT, INT, INTS, GRAPH types |
| 3 | IR versions | Track protobuf schema evolution |
| 4 | Opset compatibility | Test ops across version ranges |
| 5 | TensorProto formats | raw_data vs typed arrays |
| 6 | Graph vs subgraph | Main graph structure vs If/Loop bodies |
| Challenge | Proto-to-dict | JSON-serializable model representation |

**Next:** [Type System and Shapes](../04_Type_System_and_Shapes/)